### Install and Import necessary Libraries

In [1]:
#%pip install sqlalchemy pandas python-dotenv

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv          # environment variables
from sqlalchemy import create_engine
from pyspark.sql import SparkSession
from pyspark.sql.functions import monotonically_increasing_id
from pyspark.sql.types import StringType, IntegerType, DoubleType, LongType, TimestampType

### Extract Dataset

In [3]:
# Initialize Spark session
spark = ( 
    SparkSession.builder 
    .appName("nuga-bank_etl")               # Name of the Spark application
    .master("local[*]")                     # Use all available cores for local execution
    .config(
        "spark.jars.packages",              # List of Maven coordinates for JARs to include in the classpath
        "org.postgresql:postgresql:42.7.8"  # PostgreSQL JDBC driver version
    ) 
    .getOrCreate()                          # Create and return a SparkSession
)

spark

c:\Users\UCHE\Documents\DATA\Data-Engineering\Projects\pyspark\nuga-bank-pyspark\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [4]:
# Read the CSV file into a DataFrame
nuga_bank_df = spark.read.csv("datasets/nuga_bank_transactions.csv", header=True, inferSchema=True)

In [5]:
#show information about the dataframe 
nuga_bank_df.printSchema()

root
 |-- Transaction_Date: timestamp (nullable = true)
 |-- Amount: double (nullable = true)
 |-- Transaction_Type: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Customer_Address: string (nullable = true)
 |-- Customer_City: string (nullable = true)
 |-- Customer_State: string (nullable = true)
 |-- Customer_Country: string (nullable = true)
 |-- Company: string (nullable = true)
 |-- Job_Title: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone_Number: string (nullable = true)
 |-- Credit_Card_Number: long (nullable = true)
 |-- IBAN: string (nullable = true)
 |-- Currency_Code: string (nullable = true)
 |-- Random_Number: double (nullable = true)
 |-- Category: string (nullable = true)
 |-- Group: string (nullable = true)
 |-- Is_Active: string (nullable = true)
 |-- Last_Updated: timestamp (nullable = true)
 |-- Description: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Marital_Status: string (nullable = true)

### Transform Data

In [6]:
#copy the dataframe to another variable
nuga_bank_df_clean = nuga_bank_df

In [7]:
#show the dataframe columns
nuga_bank_df_clean.columns

['Transaction_Date',
 'Amount',
 'Transaction_Type',
 'Customer_Name',
 'Customer_Address',
 'Customer_City',
 'Customer_State',
 'Customer_Country',
 'Company',
 'Job_Title',
 'Email',
 'Phone_Number',
 'Credit_Card_Number',
 'IBAN',
 'Currency_Code',
 'Random_Number',
 'Category',
 'Group',
 'Is_Active',
 'Last_Updated',
 'Description',
 'Gender',
 'Marital_Status']

In [8]:
#show the dataframe total rows
nuga_bank_df_clean.count()

1000000

In [9]:
#total number of columns in the dataframe
len(nuga_bank_df_clean.columns)

23

In [10]:
#check the number of null values in each column
for column in nuga_bank_df_clean.columns:
    null_count = nuga_bank_df_clean.filter(nuga_bank_df_clean[column].isNull()).count()
    print(f"'{column}' Nulls: '{null_count}'")

'Transaction_Date' Nulls: '0'
'Amount' Nulls: '0'
'Transaction_Type' Nulls: '0'
'Customer_Name' Nulls: '100425'
'Customer_Address' Nulls: '100087'
'Customer_City' Nulls: '100034'
'Customer_State' Nulls: '100009'
'Customer_Country' Nulls: '100672'
'Company' Nulls: '100295'
'Job_Title' Nulls: '99924'
'Email' Nulls: '100043'
'Phone_Number' Nulls: '100524'
'Credit_Card_Number' Nulls: '100085'
'IBAN' Nulls: '100300'
'Currency_Code' Nulls: '99342'
'Random_Number' Nulls: '99913'
'Category' Nulls: '100332'
'Group' Nulls: '100209'
'Is_Active' Nulls: '100259'
'Last_Updated' Nulls: '100321'
'Description' Nulls: '100403'
'Gender' Nulls: '99767'
'Marital_Status' Nulls: '99904'


In [11]:
#fill the dataframe null values with the appropriate values
#if the datatype is timestamp, then remove the rows with null values

for column in nuga_bank_df_clean.columns:
    if isinstance(nuga_bank_df_clean.schema[column].dataType, StringType):
        nuga_bank_df_clean = nuga_bank_df_clean.fillna({column: "Unknown"})
    elif isinstance(nuga_bank_df_clean.schema[column].dataType, IntegerType):
        nuga_bank_df_clean = nuga_bank_df_clean.fillna({column: 0})
    elif isinstance(nuga_bank_df_clean.schema[column].dataType, DoubleType):
        nuga_bank_df_clean = nuga_bank_df_clean.fillna({column: 0.0})
    elif isinstance(nuga_bank_df_clean.schema[column].dataType, LongType):
        nuga_bank_df_clean = nuga_bank_df_clean.fillna({column: 0})
    elif isinstance(nuga_bank_df_clean.schema[column].dataType, TimestampType):
        nuga_bank_df_clean = nuga_bank_df_clean.dropna(subset=[column])



In [12]:
nuga_bank_df_clean.show(5)


+--------------------+------+----------------+--------------+--------------------+------------------+--------------+--------------------+--------------------+--------------------+--------------------+-------------------+------------------+--------------------+-------------+-------------+--------+-----+---------+--------------------+--------------------+-------+--------------+
|    Transaction_Date|Amount|Transaction_Type| Customer_Name|    Customer_Address|     Customer_City|Customer_State|    Customer_Country|             Company|           Job_Title|               Email|       Phone_Number|Credit_Card_Number|                IBAN|Currency_Code|Random_Number|Category|Group|Is_Active|        Last_Updated|         Description| Gender|Marital_Status|
+--------------------+------+----------------+--------------+--------------------+------------------+--------------+--------------------+--------------------+--------------------+--------------------+-------------------+------------------+---

In [13]:
#show the statistics of the dataframe
nuga_bank_df_clean.describe().show()

+-------+------------------+----------------+-------------+--------------------+-------------+--------------+----------------+-------------+------------------+-------------------+--------------------+--------------------+--------------------+-------------+------------------+--------+-------+---------+--------------------+-------+--------------+
|summary|            Amount|Transaction_Type|Customer_Name|    Customer_Address|Customer_City|Customer_State|Customer_Country|      Company|         Job_Title|              Email|        Phone_Number|  Credit_Card_Number|                IBAN|Currency_Code|     Random_Number|Category|  Group|Is_Active|         Description| Gender|Marital_Status|
+-------+------------------+----------------+-------------+--------------------+-------------+--------------+----------------+-------------+------------------+-------------------+--------------------+--------------------+--------------------+-------------+------------------+--------+-------+---------+----

In [14]:
nuga_bank_df_clean.columns

['Transaction_Date',
 'Amount',
 'Transaction_Type',
 'Customer_Name',
 'Customer_Address',
 'Customer_City',
 'Customer_State',
 'Customer_Country',
 'Company',
 'Job_Title',
 'Email',
 'Phone_Number',
 'Credit_Card_Number',
 'IBAN',
 'Currency_Code',
 'Random_Number',
 'Category',
 'Group',
 'Is_Active',
 'Last_Updated',
 'Description',
 'Gender',
 'Marital_Status']

In [15]:
#create a transaction dataframe 
transaction_df = nuga_bank_df_clean.select('transaction_Date', 'Amount', 'transaction_Type')

# add transaction_id to the transaction dataframe
transaction_df = transaction_df.withColumn("transaction_id", monotonically_increasing_id())

#reorder the columns in the transaction dataframe
transaction_df = transaction_df.select('transaction_id', 'transaction_Date', 'Amount', 'transaction_Type')

#show the transaction dataframe
transaction_df.show(5)

+--------------+--------------------+------+----------------+
|transaction_id|    transaction_Date|Amount|transaction_Type|
+--------------+--------------------+------+----------------+
|             0|2024-03-23 15:38:...| 34.76|      Withdrawal|
|             1|2024-04-22 19:15:...|163.92|      Withdrawal|
|             2|2024-04-12 19:46:...|386.32|      Withdrawal|
|             3|2024-04-17 15:29:...|407.15|         Deposit|
|             4|2024-02-10 01:51:...|161.31|         Deposit|
+--------------+--------------------+------+----------------+
only showing top 5 rows


In [16]:
#create a customer dataframe
customer_df = nuga_bank_df_clean.select('Customer_Name','Customer_Address','Customer_City','Customer_State','Customer_Country','Email','Phone_Number').distinct()

#add customer_id to the customer dataframe
customer_df = customer_df.withColumn("customer_id", monotonically_increasing_id())

#reorder the columns in the customer dataframe
customer_df = customer_df.select('customer_id','Customer_Name','Customer_Address','Customer_City','Customer_State','Customer_Country','Email','Phone_Number')

#show the customer dataframe
customer_df.show(5)


+-----------+--------------+--------------------+-------------+--------------+----------------+--------------------+--------------------+
|customer_id| Customer_Name|    Customer_Address|Customer_City|Customer_State|Customer_Country|               Email|        Phone_Number|
+-----------+--------------+--------------------+-------------+--------------+----------------+--------------------+--------------------+
|          0|Miguel Leonard|262 Beck Expressw...|      Unknown| West Virginia|         Eritrea| zweaver@example.net|             Unknown|
|          1|       Unknown|             Unknown|  Evanchester|        Oregon|         Uruguay|             Unknown| (384)778-9942x91236|
|          2|Michael Murphy|894 Williams Ridg...|Dominguezview|      New York|          Sweden|kristinstanley@ex...|+1-693-739-2204x8851|
|          3|Tina Gutierrez|    415 Taylor Knoll|    Donnastad|South Carolina|         Unknown|sarabrooks@exampl...|  623-933-0431x87174|
|          4|  Kylie Adkins|    43

In [17]:
#create a employee dataframe
employee_df = nuga_bank_df_clean.select('Company','Job_Title','Gender','Marital_Status').distinct()

#add employee_id to the employee dataframe
employee_df = employee_df.withColumn("employee_id", monotonically_increasing_id())

#reorder the columns in the employee dataframe
employee_df = employee_df.select('employee_id','Company','Job_Title','Gender','Marital_Status')

#show the employee dataframe
employee_df.show(5)

+-----------+--------------------+--------------------+-------+--------------+
|employee_id|             Company|           Job_Title| Gender|Marital_Status|
+-----------+--------------------+--------------------+-------+--------------+
|          0|         Price Group|             Unknown|   Male|        Single|
|          1|Rhodes, King and ...| Trade mark attorney|   Male|       Unknown|
|          2|Schmidt, Morgan a...|     Engineer, water| Female|        Single|
|          3|       Johnson Group|  Forensic scientist|   Male|       Unknown|
|          4|     Phillips-Prince|Production assist...|Unknown|        Single|
+-----------+--------------------+--------------------+-------+--------------+
only showing top 5 rows


In [18]:
#create a fact_table dataframe by left joining the transaction, customer and employee dataframes to the nuga_bank_df_clean dataframe
#and select the fks and the rest of the columns from the nuga_bank_df_clean dataframe that are not in the transaction, customer and employee dataframes

fact_table_df = nuga_bank_df_clean.join(transaction_df, on=['transaction_Date', 'Amount', 'transaction_Type'], how='left') \
    .join(customer_df, on=['Customer_Name','Customer_Address','Customer_City','Customer_State','Customer_Country','Email','Phone_Number'], how='left') \
    .join(employee_df, on=['Company','Job_Title','Gender','Marital_Status'], how='left') \
    .select('transaction_id', 'customer_id', 'employee_id', 'Credit_Card_Number', 'IBAN', 'Currency_Code', 'Random_Number','Category','Group','Is_Active','Last_Updated','Description')

fact_table_df.show(5)


+--------------+-----------+-----------+------------------+--------------------+-------------+-------------+--------+-----+---------+--------------------+--------------------+
|transaction_id|customer_id|employee_id|Credit_Card_Number|                IBAN|Currency_Code|Random_Number|Category|Group|Is_Active|        Last_Updated|         Description|
+--------------+-----------+-----------+------------------+--------------------+-------------+-------------+--------+-----+---------+--------------------+--------------------+
|    8589934597|42949691032|      14483|      630428157006|GB86GLHT381589496...|      Unknown|       5000.0|       A|    Z|       No|2023-10-14 00:47:...|Everything decade...|
|   17179869188|60129574164|      33729|     4021800082481|GB78UBAH195883770...|          LBP|       8097.0|       C|    X|      Yes|2023-07-11 06:06:...|Maybe teacher dee...|
|   42949672965|17179947285|      71176|    30448552436896|             Unknown|          ISK|       1108.0|       A|   

In [19]:
#output the dataframes to Parquet files in the datasets folder
transaction_df.coalesce(4).write.parquet("datasets/transaction.parquet", mode="overwrite")
customer_df.coalesce(4).write.parquet("datasets/customer.parquet", mode="overwrite")
employee_df.coalesce(4).write.parquet("datasets/employee.parquet", mode="overwrite")
fact_table_df.coalesce(4).write.parquet("datasets/fact_table.parquet", mode="overwrite")

print("DataFrames successfully written to Parquet files in the datasets folder.")

DataFrames successfully written to Parquet files in the datasets folder.


### Load dataframes to PostgreSql

In [20]:
#get the database credentials from the environment variable
load_dotenv()  # reads .env from the project folder
    
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")
driver = "org.postgresql.Driver"
url = f"jdbc:postgresql://{host}:{port}/{db_name}"

In [21]:
# create a reusable function to write the dataframes to a PostgreSQL using pyspark
def write_to_postgres(df, table_name, mode="overwrite"):
    #write the dataframe to the PostgreSQL database
    df.write \
        .format("jdbc") \
        .options(
                url=url,
                dbtable=table_name,
                user=user,
                password=password,
                driver=driver,
                batchsize="1000",
            ) \
        .mode(mode) \
        .save()

    print(f"DataFrame successfully written to PostgreSQL table '{table_name}'.")


In [22]:
# call the function to write the dataframes to the PostgreSQL database
write_to_postgres(transaction_df, "transaction")   
write_to_postgres(customer_df, "customer")
write_to_postgres(employee_df, "employee")
write_to_postgres(fact_table_df, "fact_table")

print("DataFrames successfully written to PostgreSQL database.")

DataFrame successfully written to PostgreSQL table 'transaction'.
DataFrame successfully written to PostgreSQL table 'customer'.
DataFrame successfully written to PostgreSQL table 'employee'.
DataFrame successfully written to PostgreSQL table 'fact_table'.
DataFrames successfully written to PostgreSQL database.


In [23]:
# Read the data from the PostgreSQL database into a DataFrame 

def read_from_postgres(table_name):
    #read the dataframe from the PostgreSQL database
    df = spark.read \
        .format("jdbc") \
        .options(
            url=url,
            dbtable=table_name,
            user=user,
            password=password,
            driver=driver
        ) \
        .load()

    return df   



In [24]:
# print the data from the PostgreSQL database into a DataFrame

transaction_df_from_db = read_from_postgres("transaction")
customer_df_from_db = read_from_postgres("customer")
employee_df_from_db = read_from_postgres("employee")
fact_table_df_from_db = read_from_postgres("fact_table")

transaction_df_from_db.show(5)

+--------------+--------------------+------+----------------+
|transaction_id|    transaction_Date|Amount|transaction_Type|
+--------------+--------------------+------+----------------+
|   51539607552|2024-04-05 07:35:...| 365.5|         Deposit|
|   51539607553|2024-04-09 14:11:...|608.08|         Deposit|
|   51539607554|2024-01-10 13:21:...|406.89|         Deposit|
|   51539607555|2024-01-23 02:46:...|926.36|         Deposit|
|   51539607556|2024-03-29 01:39:...| 426.4|         Deposit|
+--------------+--------------------+------+----------------+
only showing top 5 rows


In [ ]:
transaction_df_from_db.show(5)

+--------------+--------------------+------+----------------+
|transaction_id|    transaction_Date|Amount|transaction_Type|
+--------------+--------------------+------+----------------+
|   34359738492|2024-04-20 16:02:...|287.44|        Transfer|
|   34359738493|2024-01-12 16:16:...|351.01|         Deposit|
|   34359738494|2024-03-28 16:16:...|908.31|      Withdrawal|
|   34359738495|2024-04-24 01:27:...|529.82|        Transfer|
|   34359738496|2024-03-31 09:44:...|954.35|        Transfer|
+--------------+--------------------+------+----------------+
only showing top 5 rows


None